# MPA-MLF v2 — TOP 1 push

Diagnostic v1 : **le stack LogReg a tué le LB** (0.97565 sans stack → 0.96855 avec stack 50%).

v2 attaque les vrais leviers :
1. **Stack viré** du final blend
2. **ResNet-CIFAR** (archi différente) pour diversité réelle
3. **StrongCNN base=96/128** (T4 à 35% RAM, on pousse)
4. **Pseudo v2 itératif** (2 rounds)
5. **Heavy TTA** (5-way : orig + flip + shifts)
6. **Batch=512, num_workers=4** (vitesse 2×)

Budget temps T4 : ~90-100 min total. A100 : ~35 min.

In [ ]:
# === 0. Setup ===
import os
if not os.path.exists('/content/mpa'):
    !mkdir -p /content/mpa && cd /content/mpa && unzip -q /content/colab_v2.zip
!nvidia-smi | head -20
ROOT = '/content/mpa'
DATA = f'{ROOT}/data'
CSV = f'{ROOT}/csv'
SRC = f'{ROOT}/src'
OUT = f'{ROOT}/outputs'
os.makedirs(OUT, exist_ok=True)
print('Files:')
!ls $ROOT

## Phase 1 — Base models (round 1, no pseudo)

3 architectures en parallèle d'exécution séquentielle. Tout en batch=512 pour exploiter le T4.

In [ ]:
# StrongCNN base=64 × 8 seeds (workhorse, archi éprouvée)
!python $SRC/train_strongcnn.py \
  --train-csv $CSV/y_train_v2.csv \
  --sample-submission $CSV/y_test_submission_example_v2.csv \
  --train-npy $DATA/train_X.npy --test-npy $DATA/test_X.npy \
  --output-dir $OUT/scnn_b64_r1 \
  --seeds 42 1234 7 555 999 2024 31337 101 \
  --epochs 40 --batch 512 --base 64

In [ ]:
# StrongCNN base=96 × 5 seeds (bigger capacity)
!python $SRC/train_strongcnn.py \
  --train-csv $CSV/y_train_v2.csv \
  --sample-submission $CSV/y_test_submission_example_v2.csv \
  --train-npy $DATA/train_X.npy --test-npy $DATA/test_X.npy \
  --output-dir $OUT/scnn_b96_r1 \
  --seeds 42 1234 7 555 999 \
  --epochs 40 --batch 384 --base 96

In [ ]:
# ResNet-CIFAR × 6 seeds (archi différente → real diversité)
!python $SRC/train_resnet.py \
  --train-csv $CSV/y_train_v2.csv \
  --sample-submission $CSV/y_test_submission_example_v2.csv \
  --train-npy $DATA/train_X.npy --test-npy $DATA/test_X.npy \
  --output-dir $OUT/resnet_r1 \
  --seeds 42 1234 7 555 999 2024 \
  --epochs 40 --batch 384 --width-mult 1.0

## Phase 2 — First blend (no stack, pure CNN ensemble)

→ donne `probs_r1.npy` qui sert de source pour les pseudo-labels round 2.

In [ ]:
!python $SRC/blend_nostack.py \
  --sample-submission $CSV/y_test_submission_example_v2.csv \
  --probs $OUT/scnn_b64_r1/test_probs.npy $OUT/scnn_b96_r1/test_probs.npy $OUT/resnet_r1/test_probs.npy \
  --names scnn_b64 scnn_b96 resnet \
  --output-dir $OUT/blend_r1

In [ ]:
# build pseudo-label set from r1 blend
!python $SRC/make_pseudo_set.py \
  --test-probs $OUT/blend_r1/probs_mean.npy \
  --test-npy $DATA/test_X.npy \
  --output-X $OUT/pseudo_r1/X.npy \
  --output-y $OUT/pseudo_r1/y.npy \
  --threshold 0.98

## Phase 3 — Pseudo round 1 (retrain on train + high-confidence test)

In [ ]:
# StrongCNN b=64 pseudo round 1
!python $SRC/train_strongcnn.py \
  --train-csv $CSV/y_train_v2.csv \
  --sample-submission $CSV/y_test_submission_example_v2.csv \
  --train-npy $DATA/train_X.npy --test-npy $DATA/test_X.npy \
  --extra-data-npy $OUT/pseudo_r1/X.npy --extra-labels-npy $OUT/pseudo_r1/y.npy \
  --output-dir $OUT/scnn_b64_r2 \
  --seeds 42 1234 7 555 999 \
  --epochs 35 --batch 512 --base 64

In [ ]:
# StrongCNN b=96 pseudo round 1
!python $SRC/train_strongcnn.py \
  --train-csv $CSV/y_train_v2.csv \
  --sample-submission $CSV/y_test_submission_example_v2.csv \
  --train-npy $DATA/train_X.npy --test-npy $DATA/test_X.npy \
  --extra-data-npy $OUT/pseudo_r1/X.npy --extra-labels-npy $OUT/pseudo_r1/y.npy \
  --output-dir $OUT/scnn_b96_r2 \
  --seeds 42 1234 7 \
  --epochs 35 --batch 384 --base 96

In [ ]:
# ResNet pseudo round 1
!python $SRC/train_resnet.py \
  --train-csv $CSV/y_train_v2.csv \
  --sample-submission $CSV/y_test_submission_example_v2.csv \
  --train-npy $DATA/train_X.npy --test-npy $DATA/test_X.npy \
  --extra-data-npy $OUT/pseudo_r1/X.npy --extra-labels-npy $OUT/pseudo_r1/y.npy \
  --output-dir $OUT/resnet_r2 \
  --seeds 42 1234 7 555 \
  --epochs 35 --batch 384 --width-mult 1.0

## Phase 4 — Second blend + pseudo round 2 (higher threshold)

In [ ]:
!python $SRC/blend_nostack.py \
  --sample-submission $CSV/y_test_submission_example_v2.csv \
  --probs $OUT/scnn_b64_r1/test_probs.npy $OUT/scnn_b96_r1/test_probs.npy $OUT/resnet_r1/test_probs.npy \
          $OUT/scnn_b64_r2/test_probs.npy $OUT/scnn_b96_r2/test_probs.npy $OUT/resnet_r2/test_probs.npy \
  --names scnn64_r1 scnn96_r1 resnet_r1 scnn64_r2 scnn96_r2 resnet_r2 \
  --output-dir $OUT/blend_r2

In [ ]:
# Pseudo round 2 set (thr plus strict)
!python $SRC/make_pseudo_set.py \
  --test-probs $OUT/blend_r2/probs_mean.npy \
  --test-npy $DATA/test_X.npy \
  --output-X $OUT/pseudo_r2/X.npy \
  --output-y $OUT/pseudo_r2/y.npy \
  --threshold 0.995

In [ ]:
# Final round : StrongCNN b=64 pseudo v2
!python $SRC/train_strongcnn.py \
  --train-csv $CSV/y_train_v2.csv \
  --sample-submission $CSV/y_test_submission_example_v2.csv \
  --train-npy $DATA/train_X.npy --test-npy $DATA/test_X.npy \
  --extra-data-npy $OUT/pseudo_r2/X.npy --extra-labels-npy $OUT/pseudo_r2/y.npy \
  --output-dir $OUT/scnn_b64_r3 \
  --seeds 42 1234 7 555 999 \
  --epochs 30 --batch 512 --base 64

In [ ]:
# ResNet pseudo v2
!python $SRC/train_resnet.py \
  --train-csv $CSV/y_train_v2.csv \
  --sample-submission $CSV/y_test_submission_example_v2.csv \
  --train-npy $DATA/train_X.npy --test-npy $DATA/test_X.npy \
  --extra-data-npy $OUT/pseudo_r2/X.npy --extra-labels-npy $OUT/pseudo_r2/y.npy \
  --output-dir $OUT/resnet_r3 \
  --seeds 42 1234 7 \
  --epochs 30 --batch 384 --width-mult 1.0

## Phase 5 — Final mega-blend (tout ensemble, sans stack)

In [ ]:
!python $SRC/blend_nostack.py \
  --sample-submission $CSV/y_test_submission_example_v2.csv \
  --probs \
    $OUT/scnn_b64_r1/test_probs.npy \
    $OUT/scnn_b96_r1/test_probs.npy \
    $OUT/resnet_r1/test_probs.npy \
    $OUT/scnn_b64_r2/test_probs.npy \
    $OUT/scnn_b96_r2/test_probs.npy \
    $OUT/resnet_r2/test_probs.npy \
    $OUT/scnn_b64_r3/test_probs.npy \
    $OUT/resnet_r3/test_probs.npy \
  --names scnn64_r1 scnn96_r1 resnet_r1 scnn64_r2 scnn96_r2 resnet_r2 scnn64_r3 resnet_r3 \
  --output-dir $OUT/blend_final

In [ ]:
# Custom weighted blends (donner plus de poids aux modèles pseudo v2)
import numpy as np, pandas as pd
from pathlib import Path
OUT = Path('/content/mpa/outputs')
test_df = pd.read_csv('/content/mpa/csv/y_test_submission_example_v2.csv').sort_values('id').reset_index(drop=True)

probs = {
    'scnn64_r1': np.load(OUT / 'scnn_b64_r1/test_probs.npy'),
    'scnn96_r1': np.load(OUT / 'scnn_b96_r1/test_probs.npy'),
    'resnet_r1': np.load(OUT / 'resnet_r1/test_probs.npy'),
    'scnn64_r2': np.load(OUT / 'scnn_b64_r2/test_probs.npy'),
    'scnn96_r2': np.load(OUT / 'scnn_b96_r2/test_probs.npy'),
    'resnet_r2': np.load(OUT / 'resnet_r2/test_probs.npy'),
    'scnn64_r3': np.load(OUT / 'scnn_b64_r3/test_probs.npy'),
    'resnet_r3': np.load(OUT / 'resnet_r3/test_probs.npy'),
}

# Weighted : les r3 pseudo-v2 pèsent le plus (meilleure qualité), r1 le moins
W = {
    'scnn64_r1': 1.0, 'scnn96_r1': 1.0, 'resnet_r1': 1.0,
    'scnn64_r2': 1.5, 'scnn96_r2': 1.5, 'resnet_r2': 1.5,
    'scnn64_r3': 2.0, 'resnet_r3': 2.0,
}
total_w = sum(W.values())
weighted = sum(probs[k] * W[k] for k in W) / total_w

# mix arch (pondérer les 3 archis ensemble)
scnn_avg = (probs['scnn64_r1']+probs['scnn96_r1']+probs['scnn64_r2']+probs['scnn96_r2']+probs['scnn64_r3'])/5
resnet_avg = (probs['resnet_r1']+probs['resnet_r2']+probs['resnet_r3'])/3
arch_balanced = 0.5 * scnn_avg + 0.5 * resnet_avg

# r3 only (last round)
r3_only = 0.5 * probs['scnn64_r3'] + 0.5 * probs['resnet_r3']

# r2 + r3 (exclude r1)
r2r3 = (probs['scnn64_r2']+probs['scnn96_r2']+probs['resnet_r2']+probs['scnn64_r3']+probs['resnet_r3'])/5

out_sub = OUT / 'submissions_final'; out_sub.mkdir(exist_ok=True)
for tag, pr in [('weighted', weighted), ('arch_balanced', arch_balanced),
                ('r3_only', r3_only), ('r2r3', r2r3)]:
    sub = test_df.copy()
    sub['target'] = pr.argmax(1).astype(int)
    path = out_sub / f'submission_{tag}.csv'
    sub.to_csv(path, index=False)
    np.save(out_sub / f'probs_{tag}.npy', pr)
    print(tag, path, np.bincount(pr.argmax(1), minlength=4))

In [ ]:
# Zip + download
!cd /content/mpa/outputs && zip -r /content/results_v2.zip blend_r1 blend_r2 blend_final submissions_final pseudo_r1/y.npy pseudo_r2/y.npy
from google.colab import files
files.download('/content/results_v2.zip')

## Tips

- Sur A100 : passe `--batch 768` partout, divise le temps par 2-3.
- Les submissions à tester en priorité (après download) :
  1. `blend_final/submission_mean.csv` — mean des 8 sources
  2. `submissions_final/submission_weighted.csv` — poids par round
  3. `blend_final/submission_geom.csv` — géométrique
  4. `submissions_final/submission_arch_balanced.csv`
  5. `submissions_final/submission_r3_only.csv`